# ForexFactory Scraper for Economic Event Surprises

**Project:** Economic Event-Driven Forex Trading Assistant (FYP)

**Purpose:** Scrape the ForexFactory economic calendar to extract analyst consensus forecasts and actual released values for both USD and EUR macroeconomic events. The difference (actual minus forecast) provides the market surprise used as a feature in the main ML pipeline.

**Coverage:** January 2019 to December 2025.

**Events captured:**
- **USD:** CPI m/m, Non-Farm Employment Change (NFP), Federal Funds Rate (FOMC)
- **EUR:** Main Refinancing Rate (ECB), CPI Flash Estimate y/y, Core CPI Flash Estimate y/y

**Output:** Six CSV files, each containing two columns (`date`, `<event>_surprise`):
- `cpi_surprise.csv`, `nfp_surprise.csv`, `fomc_surprise.csv`
- `ecb_rate_surprise.csv`, `eu_cpi_surprise.csv`, `eu_core_cpi_surprise.csv`

**Technical note:** ForexFactory uses Cloudflare bot protection. The `cloudscraper` library bypasses this by emulating a browser fingerprint, allowing programmatic access without requiring API authentication.

In [1]:
# install dependencies (run once, then comment out)
# %pip install cloudscraper beautifulsoup4 pandas

import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
import time

# initialise cloudscraper session (handles Cloudflare bypass)
scraper = cloudscraper.create_scraper()

print("Setup complete")

Setup complete


## Section 1: Scraper Function

A single parameterised scraper function handles both USD and EUR events. The function takes:
- `year` and `month`: which calendar month to scrape
- `currency`: filter to a specific currency code (e.g. 'USD' or 'EUR')
- `event_keywords`: list of event name substrings to keep

This design avoids code duplication and makes it trivial to extend to additional currencies (e.g. GBP from the Bank of England) in future work.

In [2]:
def scrape_forexfactory(year, month, currency, event_keywords):
    """
    Scrape ForexFactory economic calendar for one month.

    Parameters
    ----------
    year : int
        Year to scrape (e.g. 2024).
    month : int
        Month number (1-12).
    currency : str
        ISO currency code to filter on (e.g. 'USD', 'EUR').
    event_keywords : list of str
        Event name substrings to keep (e.g. ['CPI', 'Non-Farm', 'Fed']).
        A row is kept if any keyword appears in the event name.

    Returns
    -------
    pandas.DataFrame
        Columns: date, year, month, currency, event, actual, forecast, previous.
    """
    month_names = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
                   'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
    url = f"https://www.forexfactory.com/calendar?month={month_names[month-1]}.{year}"

    response = scraper.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    rows = soup.find_all('tr', class_='calendar__row')

    results = []
    current_date = None

    for row in rows:
        # ForexFactory only shows the date on the first row of each day
        date_cell = row.find('td', class_='calendar__date')
        if date_cell and date_cell.text.strip():
            current_date = date_cell.text.strip()

        event_cell = row.find('td', class_='calendar__event')
        if not event_cell:
            continue
        event_name = event_cell.text.strip()

        # currency filter
        currency_cell = row.find('td', class_='calendar__currency')
        if not currency_cell or currency_cell.text.strip() != currency:
            continue

        # event keyword filter
        if not any(k in event_name for k in event_keywords):
            continue

        actual = row.find('td', class_='calendar__actual')
        forecast = row.find('td', class_='calendar__forecast')
        previous = row.find('td', class_='calendar__previous')

        results.append({
            'date':     current_date,
            'year':     year,
            'month':    month,
            'currency': currency,
            'event':    event_name,
            'actual':   actual.text.strip()   if actual   else '',
            'forecast': forecast.text.strip() if forecast else '',
            'previous': previous.text.strip() if previous else '',
        })

    return pd.DataFrame(results)


# quick smoke test on one month
test_df = scrape_forexfactory(2024, 1, 'USD', ['CPI', 'Non-Farm', 'Fed'])
print(f"Smoke test (Jan 2024 USD): {len(test_df)} rows found")
test_df.head()

Smoke test (Jan 2024 USD): 9 rows found


,date,year,month,currency,event,actual,forecast,previous
0,Thu Jan 4,2024,1,USD,ADP Non-Farm Employment Change,164K,120K,101K
1,Fri Jan 5,2024,1,USD,Non-Farm Employment Change,216K,168K,173K
2,Thu Jan 11,2024,1,USD,Core CPI m/m,0.3%,0.3%,0.3%
3,Thu Jan 11,2024,1,USD,Core CPI y/y,3.9%,3.8%,4.0%
4,Thu Jan 11,2024,1,USD,CPI m/m,0.3%,0.2%,0.1%


## Section 2: Run the Scrape

Loop through every month from January 2019 to December 2025 for both currencies. The `time.sleep(3)` between requests avoids triggering ForexFactory's rate-limit / Cloudflare re-challenge.

Expected runtime: approximately 4-5 minutes per currency (84 months × 3 seconds).

In [3]:
# USD events: CPI, Non-Farm Payrolls, Federal Funds Rate
usd_keywords = ['CPI', 'Non-Farm', 'NFP', 'Fed', 'Interest Rate']
usd_data = []

for year in range(2019, 2026):
    for month in range(1, 13):
        print(f"USD {year}-{month:02d}...", end=' ')
        try:
            df_m = scrape_forexfactory(year, month, 'USD', usd_keywords)
            usd_data.append(df_m)
            print(f"{len(df_m)} events")
            time.sleep(3)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

usd_raw = pd.concat(usd_data, ignore_index=True)
print(f"\nTotal USD events scraped: {len(usd_raw)}")
print(usd_raw['event'].value_counts())

USD 2019-01... 9 events
USD 2019-02... 10 events
USD 2019-03... 10 events
USD 2019-04... 8 events
USD 2019-05... 9 events
USD 2019-06... 10 events
USD 2019-07... 12 events
USD 2019-08... 8 events
USD 2019-09... 9 events
USD 2019-10... 10 events
USD 2019-11... 9 events
USD 2019-12... 9 events
USD 2020-01... 9 events
USD 2020-02... 13 events
USD 2020-03... 18 events
USD 2020-04... 11 events
USD 2020-05... 9 events
USD 2020-06... 13 events
USD 2020-07... 11 events
USD 2020-08... 8 events
USD 2020-09... 13 events
USD 2020-10... 8 events
USD 2020-11... 9 events
USD 2020-12... 11 events
USD 2021-01... 9 events
USD 2021-02... 11 events
USD 2021-03... 13 events
USD 2021-04... 8 events
USD 2021-05... 8 events
USD 2021-06... 11 events
USD 2021-07... 11 events
USD 2021-08... 8 events
USD 2021-09... 11 events
USD 2021-10... 8 events
USD 2021-11... 10 events
USD 2021-12... 10 events
USD 2022-01... 10 events
USD 2022-02... 10 events
USD 2022-03... 12 events
USD 2022-04... 7 events
USD 2022-05... 10 

In [4]:
# EUR events: ECB Main Refinancing Rate, CPI Flash Estimate (regular and Core)
eur_keywords = ['Main Refinancing Rate', 'CPI Flash Estimate']
eur_data = []

for year in range(2019, 2026):
    for month in range(1, 13):
        print(f"EUR {year}-{month:02d}...", end=' ')
        try:
            df_m = scrape_forexfactory(year, month, 'EUR', eur_keywords)
            eur_data.append(df_m)
            print(f"{len(df_m)} events")
            time.sleep(3)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

eur_raw = pd.concat(eur_data, ignore_index=True)
print(f"\nTotal EUR events scraped: {len(eur_raw)}")
print(eur_raw['event'].value_counts())

EUR 2019-01... 3 events
EUR 2019-02... 2 events
EUR 2019-03... 3 events
EUR 2019-04... 3 events
EUR 2019-05... 2 events
EUR 2019-06... 5 events
EUR 2019-07... 3 events
EUR 2019-08... 2 events
EUR 2019-09... 1 events
EUR 2019-10... 5 events
EUR 2019-11... 2 events
EUR 2019-12... 1 events
EUR 2020-01... 5 events
EUR 2020-02... 0 events
EUR 2020-03... 5 events
EUR 2020-04... 3 events
EUR 2020-05... 2 events
EUR 2020-06... 3 events
EUR 2020-07... 3 events
EUR 2020-08... 0 events
EUR 2020-09... 3 events
EUR 2020-10... 5 events
EUR 2020-11... 0 events
EUR 2020-12... 3 events
EUR 2021-01... 3 events
EUR 2021-02... 2 events
EUR 2021-03... 5 events
EUR 2021-04... 3 events
EUR 2021-05... 0 events
EUR 2021-06... 5 events
EUR 2021-07... 3 events
EUR 2021-08... 2 events
EUR 2021-09... 1 events
EUR 2021-10... 5 events
EUR 2021-11... 2 events
EUR 2021-12... 1 events
EUR 2022-01... 2 events
EUR 2022-02... 3 events
EUR 2022-03... 3 events
EUR 2022-04... 5 events
EUR 2022-05... 2 events
EUR 2022-06... 1

## Section 3: Data Cleaning

Two cleaning steps:

1. **Number cleaning**: ForexFactory formats values with suffixes (`%`, `K`, `M`) and qualifiers (`<`). Strip these and convert to float. Compute `surprise = actual - forecast`.

2. **Date parsing**: ForexFactory's date format is `Day Mon DD` (e.g. `Fri Jan 12`) without a year on the page. Combine with the scraped year, strip the weekday prefix, and parse to a proper datetime.

In [5]:
def clean_numbers(df):
    """
    Clean ForexFactory numeric columns and compute surprise.

    Strips %, K, M, < symbols from actual / forecast / previous columns,
    converts to float (NaN on failure), and adds a `surprise` column
    computed as actual minus forecast.
    """
    df = df.copy()
    for col in ['actual', 'forecast', 'previous']:
        if col not in df.columns:
            continue
        df[col] = (
            df[col].astype(str)
                   .str.replace('%', '', regex=False)
                   .str.replace('K', '', regex=False)
                   .str.replace('M', '', regex=False)
                   .str.replace('<', '', regex=False)
                   .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['surprise'] = df['actual'] - df['forecast']
    return df


def parse_dates(df):
    """
    Parse ForexFactory date strings into proper datetime values.

    Strips weekday prefix, appends year, handles missing space between
    month and day, and parses with the format 'Mon DD YYYY'.
    Rows with un-parseable dates are dropped.
    """
    df = df.copy()
    # strip weekday prefix (e.g. 'Fri Jan 12' -> 'Jan 12')
    df['date_clean'] = df['date'].astype(str).str.replace(
        r'(Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+', '', regex=True
    ).str.strip()
    # append the year
    df['date_full'] = df['date_clean'] + ' ' + df['year'].astype(str)
    # insert a space between text and digits (handles 'May23' edge case)
    df['date_full'] = df['date_full'].str.replace(
        r'([A-Za-z])([0-9])', r'\1 \2', regex=True
    )
    # collapse multiple spaces
    df['date_full'] = df['date_full'].str.replace(r'\s+', ' ', regex=True).str.strip()
    # parse to datetime
    df['date'] = pd.to_datetime(df['date_full'], format='%b %d %Y', errors='coerce')
    return df


# apply both cleaning steps to USD and EUR raw data
usd_clean = parse_dates(clean_numbers(usd_raw)).dropna(subset=['date', 'surprise'])
eur_clean = parse_dates(clean_numbers(eur_raw)).dropna(subset=['date', 'surprise'])

print(f"USD events after cleaning: {len(usd_clean)} "
      f"({usd_clean['date'].min().date()} to {usd_clean['date'].max().date()})")
print(f"EUR events after cleaning: {len(eur_clean)} "
      f"({eur_clean['date'].min().date()} to {eur_clean['date'].max().date()})")

USD events after cleaning: 634 (2019-01-03 to 2025-12-18)
EUR events after cleaning: 224 (2019-01-04 to 2025-12-18)


## Section 4: Filter and Save USD CSVs

Three USD CSVs corresponding to the three US economic releases used in the main pipeline. The event-name filter is strict (exact match on the canonical ForexFactory name) to avoid contamination from non-US events that share keywords (e.g. French CPI also contains 'CPI' in the name).

In [6]:
# USD CPI m/m
cpi_df = usd_clean[usd_clean['event'] == 'CPI m/m'][['date', 'surprise']].copy()
cpi_df.columns = ['date', 'cpi_surprise']
cpi_df = cpi_df.sort_values('date').reset_index(drop=True)
cpi_df.to_csv('cpi_surprise.csv', index=False)
print(f"USD CPI saved: {len(cpi_df)} events")

# Non-Farm Payrolls
nfp_df = usd_clean[usd_clean['event'] == 'Non-Farm Employment Change'][['date', 'surprise']].copy()
nfp_df.columns = ['date', 'nfp_surprise']
nfp_df = nfp_df.sort_values('date').reset_index(drop=True)
nfp_df.to_csv('nfp_surprise.csv', index=False)
print(f"USD NFP saved: {len(nfp_df)} events")

# Federal Funds Rate (FOMC)
fomc_df = usd_clean[usd_clean['event'] == 'Federal Funds Rate'][['date', 'surprise']].copy()
fomc_df.columns = ['date', 'fomc_surprise']
fomc_df = fomc_df.sort_values('date').reset_index(drop=True)
fomc_df.to_csv('fomc_surprise.csv', index=False)
print(f"USD FOMC saved: {len(fomc_df)} events")

USD CPI saved: 82 events
USD NFP saved: 83 events
USD FOMC saved: 55 events


## Section 5: Filter and Save ECB / EUR CSVs

Three EUR CSVs corresponding to the three Euro-area economic releases. Added in response to mid-review supervisor feedback recommending coverage of both sides of the EUR/USD pair.

In [7]:
# ECB Main Refinancing Rate
ecb_rate = eur_clean[eur_clean['event'] == 'Main Refinancing Rate'][['date', 'surprise']].copy()
ecb_rate.columns = ['date', 'ecb_rate_surprise']
ecb_rate = ecb_rate.sort_values('date').reset_index(drop=True)
ecb_rate.to_csv('ecb_rate_surprise.csv', index=False)
print(f"ECB Rate saved: {len(ecb_rate)} events")

# EU CPI Flash Estimate
eu_cpi = eur_clean[eur_clean['event'] == 'CPI Flash Estimate y/y'][['date', 'surprise']].copy()
eu_cpi.columns = ['date', 'eu_cpi_surprise']
eu_cpi = eu_cpi.sort_values('date').reset_index(drop=True)
eu_cpi.to_csv('eu_cpi_surprise.csv', index=False)
print(f"EU CPI saved: {len(eu_cpi)} events")

# EU Core CPI Flash Estimate
eu_core_cpi = eur_clean[eur_clean['event'] == 'Core CPI Flash Estimate y/y'][['date', 'surprise']].copy()
eu_core_cpi.columns = ['date', 'eu_core_cpi_surprise']
eu_core_cpi = eu_core_cpi.sort_values('date').reset_index(drop=True)
eu_core_cpi.to_csv('eu_core_cpi_surprise.csv', index=False)
print(f"EU Core CPI saved: {len(eu_core_cpi)} events")

ECB Rate saved: 56 events
EU CPI saved: 84 events
EU Core CPI saved: 84 events


## Section 6: Sanity Checks

Distribution of surprise values for each event. Useful for verifying scraper output and for the methodology discussion in the final report.

Note: ECB Rate surprises are expected to be near-zero in most observations due to the ECB's forward guidance policy framework (decisions are telegraphed via speeches and meeting minutes weeks in advance, leaving little room for surprise on the announcement day). This is consistent with the macroeconomic announcement literature.

In [8]:
datasets = [
    ('USD CPI',     cpi_df,       'cpi_surprise'),
    ('USD NFP',     nfp_df,       'nfp_surprise'),
    ('USD FOMC',    fomc_df,      'fomc_surprise'),
    ('ECB Rate',    ecb_rate,     'ecb_rate_surprise'),
    ('EU CPI',      eu_cpi,       'eu_cpi_surprise'),
    ('EU Core CPI', eu_core_cpi,  'eu_core_cpi_surprise'),
]

for name, df, col in datasets:
    stats = df[col].describe()
    non_zero = (df[col] != 0).sum()
    print(f"{name}: n={len(df)}, mean={stats['mean']:.3f}, std={stats['std']:.3f}, "
          f"min={stats['min']:.3f}, max={stats['max']:.3f}, non-zero={non_zero}/{len(df)}")

USD CPI: n=82, mean=0.027, std=0.139, min=-0.200, max=0.600, non-zero=55/82
USD NFP: n=83, mean=168.892, std=1161.094, min=-724.000, max=10259.000, non-zero=82/83
USD FOMC: n=55, mean=0.000, std=0.048, min=-0.250, max=0.250, non-zero=2/55
ECB Rate: n=56, mean=0.009, std=0.047, min=0.000, max=0.250, non-zero=2/56
EU CPI: n=84, mean=0.042, std=0.226, min=-0.500, max=0.800, non-zero=58/84
EU Core CPI: n=84, mean=0.033, std=0.160, min=-0.500, max=0.500, non-zero=57/84
